# 00 — Data Tour
Phase-1 record: what each raw dataset looks like on disk. Companion to `phasewise_notes.txt`.
Run from `notebooks/` — paths are relative to the repo root.

In [1]:
import json, csv
from pathlib import Path
from collections import Counter
import pandas as pd
import duckdb

RAW = Path("../data/raw")
SB, WY, TM = RAW/"statsbomb/data", RAW/"wyscout", RAW/"transfermarkt"
pd.set_option("display.width", 160)

## 1. StatsBomb — corpus overview
One JSON per match. 120×80 pitch, events oriented so the acting team attacks toward x=120.

In [2]:
comps = json.load(open(SB/"competitions.json"))
df = pd.DataFrame(comps)[["competition_id","season_id","competition_name","season_name","competition_gender"]]
print(f"{len(comps)} competition-seasons | {len(list((SB/'events').glob('*.json')))} event files on disk")
df.groupby("competition_name").size().sort_values(ascending=False)

80 competition-seasons | 4235 event files on disk


competition_name
Champions League           18
La Liga                    18
FIFA World Cup              8
FA Women's Super League     4
Ligue 1                     3
Copa del Rey                3
1. Bundesliga               2
UEFA Women's Euro           2
UEFA Euro                   2
Serie A                     2
Premier League              2
NWSL                        2
Liga Profesional            2
Women's World Cup           2
African Cup of Nations      1
Major League Soccer         1
Liga F                      1
North American League       1
Indian Super league         1
Frauen Bundesliga           1
Serie A Women               1
FIFA U20 World Cup          1
UEFA Europa League          1
Copa America                1
dtype: int64

## 1b. StatsBomb — event anatomy (WC2022 final, Argentina v France)

In [3]:
ev = json.load(open(SB/"events/3869685.json"))
print(len(ev), "events. Type census:")
Counter(e["type"]["name"] for e in ev).most_common(15)

4407 events. Type census:


[('Pass', 1263),
 ('Ball Receipt*', 1114),
 ('Carry', 940),
 ('Pressure', 361),
 ('Ball Recovery', 115),
 ('Duel', 98),
 ('Dribble', 54),
 ('Block', 50),
 ('Foul Committed', 48),
 ('Clearance', 45),
 ('Foul Won', 44),
 ('Goal Keeper', 44),
 ('Shot', 38),
 ('Miscontrol', 35),
 ('Dispossessed', 34)]

In [4]:
# the four event types this project lives on
def first(t, cond=lambda e: True):
    return next(e for e in ev if e["type"]["name"]==t and cond(e))

p = first("Pass", lambda e: "end_location" in e.get("pass", {}))
c = first("Carry")
d = first("Dribble")
s = first("Shot", lambda e: "freeze_frame" in e.get("shot", {}))

print("PASS   ", p["player"]["name"], p["location"], "->", p["pass"]["end_location"], p["pass"]["height"]["name"])
print("CARRY  ", c["player"]["name"], c["location"], "->", c["carry"]["end_location"], f"{c.get('duration',0):.2f}s")
print("DRIBBLE", d["player"]["name"], "at", d["location"], "outcome:", d["dribble"]["outcome"]["name"])
print("SHOT   ", s["player"]["name"], s["location"], "-> (x,y,z)", s["shot"]["end_location"],
      "xG", round(s["shot"]["statsbomb_xg"],3), "| freeze_frame:", len(s["shot"]["freeze_frame"]), "players")

PASS    Antoine Griezmann [61.0, 40.1] -> [48.0, 43.2] Ground Pass
CARRY   Aurélien Djani Tchouaméni [48.0, 43.2] -> [49.7, 43.6] 1.17s
DRIBBLE Ángel Fabián Di María Hernández at [108.9, 4.9] outcome: Incomplete
SHOT    Alexis Mac Allister [92.4, 30.0] -> (x,y,z) [117.3, 38.3, 0.8] xG 0.025 | freeze_frame: 14 players


## 2. Wyscout — event anatomy
One JSON per **competition**. Coordinates are 0–100 percentages (acting team → x=100).
Everything is numeric IDs — names need the lookup tables. **No carry event type exists.**

In [5]:
wev = json.load(open(WY/"events/events_World_Cup.json"))
print(len(wev), "WC2018 events. One raw event:")
wev[100]

101759 WC2018 events. One raw event:


{'eventId': 8,
 'subEventName': 'Simple pass',
 'tags': [{'id': 1801}],
 'playerId': 122832,
 'positions': [{'y': 76, 'x': 69}, {'y': 91, 'x': 64}],
 'matchId': 2057954,
 'eventName': 'Pass',
 'teamId': 16521,
 'matchPeriod': '1H',
 'eventSec': 296.998921,
 'subEventId': 85,
 'id': 258612225}

In [6]:
print("event types:", Counter(e["eventName"] for e in wev).most_common())
print()
print("sub-events (take-ons hide in 'Ground attacking duel'):")
Counter(e["subEventName"] for e in wev if e["subEventName"]).most_common(15)

event types: [('Pass', 56457), ('Duel', 25927), ('Others on the ball', 9282), ('Free Kick', 5964), ('Foul', 1766), ('Shot', 1419), ('Save attempt', 560), ('Goalkeeper leaving line', 212), ('Offside', 172)]

sub-events (take-ons hide in 'Ground attacking duel'):


[('Simple pass', 45048),
 ('Ground attacking duel', 8072),
 ('Ground defending duel', 8007),
 ('Touch', 6403),
 ('Air duel', 5518),
 ('Ground loose ball duel', 4330),
 ('High pass', 4040),
 ('Head pass', 2918),
 ('Throw in', 2607),
 ('Clearance', 1931),
 ('Cross', 1887),
 ('Foul', 1634),
 ('Shot', 1419),
 ('Launch', 1238),
 ('Free Kick', 1199)]

In [7]:
# decode numeric tags -> names
tags = pd.read_csv(WY/"tags2name.csv")
players = {p["wyId"]: p["shortName"] for p in json.load(open(WY/"players.json"))}
e = wev[100]
print("player:", players.get(e["playerId"]), "| tags:",
      [tags.loc[tags.Tag==t["id"], "Description"].values for t in e["tags"]])

player: Salem Al Dawsari | tags: [array(['Accurate'], dtype=object)]


## 3. Transfermarkt — labels & metadata (12 CSVs)
No events. Quality axis (valuations), E5 pairs (transfers), bio (players), shrinkage minutes (appearances).
Note: `game_lineups.csv` has malformed rows — always read with `ignore_errors=true`.

In [8]:
con = duckdb.connect()
for tbl in ["players","player_valuations","transfers","appearances"]:
    q = f"read_csv('{TM}/{tbl}.csv', ignore_errors=true)"
    n = con.execute(f"SELECT COUNT(*) FROM {q}").fetchone()[0]
    print(f"=== {tbl} — {n:,} rows")
    display(con.execute(f"SELECT * FROM {q} LIMIT 3").df())

=== players — 50,149 rows


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,<NA>,<NA>,<NA>,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000,30000000
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,<NA>,<NA>,<NA>,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000,8000000
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,<NA>,<NA>,<NA>,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000,34500000


=== player_valuations — 656,301 rows


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1


=== transfers — 175,165 rows


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,645842,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,150000.0,Chan-gi An
2,677470,2028-02-02,27/28,6505,3535,Gimcheon Sangmu,Ulsan HD,0.0,400000.0,Yool Heo


=== appearances — 1,894,350 rows


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45


In [9]:
# caveat check: future-dated transfers exist (pre-announced deals) — filter before use
con.execute(f"""SELECT COUNT(*) AS future_rows, MAX(transfer_date) AS max_date
FROM read_csv('{TM}/transfers.csv', ignore_errors=true)
WHERE transfer_date > CURRENT_DATE""").df()

,future_rows,max_date
0,496,2030-06-30


## Caveats recap
1. Both providers: coordinates oriented per acting team — never flip by half.
2. StatsBomb: 80 comp-seasons on disk; ingest an explicit include-list, not everything.
3. Wyscout: no carries (SPADL inference later); dribbles inside Duel + tags.
4. Three ID universes; joins need reep (pending download).
5. Clocks reset per period (SB) / per half (Wyscout).
6. Transfermarkt: future transfer dates, malformed lineup rows, irregular valuation snapshots.